### Molecular dynamics trajectory analysis and reporting tool using MDTraj

This Python script uses the `MDTraj` and `pandas` libraries to automate the analysis of molecular dynamics trajectories. It allows users to select specific calculations, such as bond length, angle, dihedral, and RMSD, and define the atom indices for these computations. The script then iterates through trajectory files in a specified directory, performs the chosen calculations, and consolidates all results into a single, organized Excel spreadsheet.

In [ ]:
''' To run the script, you will need the following files:

For each trajectory folder: a trajectory file named of .xyz type (the exact name depending on the specific
code used to run the AIMD simulation) and a topology .pdb file. A reference structure: reference_structure.xyz
and reference_structure.pdb if you want to perform RMSD. '''

# The following Python libraries and modules are required

import pandas as pd
import numpy as np
import mdtraj as md
from pathlib import Path

# The following code snippet is used to keep track of which calculations to do
# selected_calculations[0] = bond length
# selected_calculations[1] = angle
# selected_calculations[2] = dihedral
# selected_calculations[3] = rmsd
# Values will be changed to '1' based on user input

print("Please choose one or more calculations:")
print("1. Bond")
print("2. Angle")
print("3. Dihedral")
print("4. RMSD")

user_input = input("Enter the numbers of the calculations you want to perform (separated by commas and no spaces): ")
calculation_numbers = [int(num.strip()) for num in user_input.split(",")]

# Initialize a tracker array to all zeros for selected calculations
selected_calculations = [0, 0, 0, 0]
for num in calculation_numbers:
    if num == 1:
        selected_calculations[0] = 1
    if num == 2:
        selected_calculations[1] = 1
    if num == 3:
        selected_calculations[2] = 1
    if num == 4:
        selected_calculations[3] = 1

# Display the user's choices as confirmation
print("You have chosen the following calculations:")
if selected_calculations[0] == 1:
    print("- Bond")
if selected_calculations[1] == 1:
    print("- Angle")
if selected_calculations[2] == 1:
    print("- Dihedral")
if selected_calculations[3] == 1:
    print("- RMSD")

# Declare and initialize indices arrays with specified default sizes
bond_indices = [0, 0]
angle_indices = [0, 0, 0]
dihedral_indices = [0, 0, 0, 0]

# Prompt user for specific indices based on selected calculations
if selected_calculations[0] == 1:
    # Store bond indices into an array
    print("Specify the two atom indices for bond length calculation: ")
    for i in range (0, len(bond_indices)):
        a = int(input())
        bond_indices[i] = a
if selected_calculations[1] == 1:
    # Store angle indices into an array
    print("Specify the three atom indices for angle calculation: ")
    for i in range (0, len(angle_indices)):
        a = int(input())
        angle_indices[i] = a
if selected_calculations[2] == 1:
    # Store dihedral indices into an array
    print("Specify the four atom indices for dihedral calculation: ")
    for i in range (0, len(dihedral_indices)):
        a = int(input())
        dihedral_indices[i] = a
if selected_calculations[3] == 1:
    print("RMSD will be calculated against the reference structure.")

# Ask user for the desired output file name
excelSheetName = input("What would you like the output file to be called? ")

# Define the directory containing the trajectory files.
# The 'r' prefix creates a raw string literal to handle backslashes correctly.
directory=Path(r"C:\trajectories")

# Create a Pandas Excel writer object, specifying the output file name and engine.
writer = pd.ExcelWriter(excelSheetName + ".xlsx", engine="xlsxwriter")

# Load the reference structure for RMSD calculation
reference = md.load_xyz('reference_structure.xyz', top='reference_structure.pdb')

# Initialize a counter for tracking the starting column when writing to the Excel file
y=0

# Iterate through all subdirectories within the specified directory
for files in directory.iterdir():

    # Process only if the current item is a directory (representing a trajectory)
    if files.is_dir():

        ''' Construct the full path to the "View.xyz" file inside the subdirectory.
        The name of this file will depend on the naming scheme for the trajectory file
        of the code you are using to run the molecular dynamics simulation '''
        file_xyz=files.joinpath("View.xyz")

        # Convert the WindowsPath object to a string for use with mdtraj.load_xyz
        string_file_xyz=str(file_xyz)

        # Load a molecular trajectory from the .xyz file, using 'structure.pdb' as topology
        traj = md.load_xyz(string_file_xyz, top='structure.pdb')

        # Increment the counter for each successfully loaded trajectory
        y=y+1

        # Perform calculations based on user's choices
        if selected_calculations[0]==1 :
            # Calculate bond length for the specified indices
            length = md.compute_distances(traj, [bond_indices])
            # Create a DataFrame from the calculated bond lengths
            dl = pd.DataFrame(length)
            # Write the DataFrame to an Excel sheet named "Bond Length"
            dl.to_excel(writer, sheet_name="Bond Length", startrow = 0, startcol = y-1, index=False)

        if selected_calculations[1]==1 :
            # Calculate angle for the specified indices
            angle = md.compute_angles(traj, [angle_indices])
            # Create a DataFrame from the calculated angles
            da = pd.DataFrame(angle)
            # Write the DataFrame to an Excel sheet named "Angle"
            da.to_excel(writer, sheet_name="Angle", startrow = 0, startcol = y-1, index=False)

        if selected_calculations[2]==1 :
            # Calculate dihedral angle for the specified indices
            dihedral = md.compute_dihedrals(traj, [dihedral_indices])
            # Create a DataFrame from the calculated dihedral angles
            dd = pd.DataFrame(dihedral)
            # Write the DataFrame to an Excel sheet named "Dihedral"
            dd.to_excel(writer, sheet_name="Dihedral", startrow = 0, startcol = y-1, index=False)

        if selected_calculations[3]==1 :
            # Calculate RMSD of the trajectory against the reference structure
            rmsd = md.rmsd(traj, reference)
            # Create a DataFrame from the calculated RMSD values
            drmsd = pd.DataFrame(rmsd)
            # Write the DataFrame to an Excel sheet named "RMSD"
            drmsd.to_excel(writer, sheet_name="RMSD", startrow = 0, startcol = y-1, index=False)

# Ensure the Excel writer is closed to save the file correctly
writer.close()

print("Your calculations were a success!")